In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

In [0]:
%sql
-- Check for NULL values
SELECT 
  'NULL Check' as check_type,
  COUNT(*) as total_rows,
  SUM(CASE WHEN game_name IS NULL THEN 1 ELSE 0 END) as null_game_name,
  SUM(CASE WHEN genre IS NULL THEN 1 ELSE 0 END) as null_genre,
  SUM(CASE WHEN rank_type IS NULL THEN 1 ELSE 0 END) as null_rank_type,
  SUM(CASE WHEN rank IS NULL THEN 1 ELSE 0 END) as null_rank
FROM bronze_rankings;

In [0]:
%sql
-- Check for empty strings
SELECT 
  'Empty String Check' as check_type,
  SUM(CASE WHEN TRIM(game_name) = '' THEN 1 ELSE 0 END) as empty_game_name,
  SUM(CASE WHEN TRIM(genre) = '' THEN 1 ELSE 0 END) as empty_genre,
  SUM(CASE WHEN TRIM(rank_type) = '' THEN 1 ELSE 0 END) as empty_rank_type
FROM bronze_rankings;

In [0]:
%sql
-- check granularity of dataset and whether there are any duplicates
SELECT game_name, genre, rank_type, COUNT(*) AS n
FROM bronze_rankings
GROUP BY game_name, genre, rank_type
HAVING COUNT(*) > 1

In [0]:
%sql
-- check distinct value of categoricals to make sure there's nothing unexpected
SELECT rank_type, COUNT(*) AS n, COUNT(DISTINCT game_name) AS games, COUNT(DISTINCT genre) AS genres
FROM bronze_rankings
GROUP BY rank_type

In [0]:
%sql
-- check join coverage againest silver_games for downstream analytics
SELECT
  COUNT(*) AS total_rank_rows,
  COUNT(g.game_key) AS matched_rows,
  COUNT(*) - COUNT(g.game_key) AS unmatched_rows
FROM bronze_rankings r
LEFT JOIN silver_games g
  ON LOWER(TRIM(REGEXP_REPLACE(r.game_name, '[®™©]', ''))) = g.game_key

In [0]:
%sql
CREATE OR REPLACE TABLE silver_rankings AS
SELECT
  LOWER(TRIM(REGEXP_REPLACE(game_name, '[®™©]', ''))) AS game_key,
  game_name,
  genre,
  rank_type,
  rank
FROM bronze_rankings